In [7]:
import os
from google.colab import drive

!rm -rf /content/drive/* /content/drive/.[!.]* 2>/dev/null || true
drive.mount('/content/drive')

drive_dir = '/content/drive/MyDrive/otto_project'
os.makedirs(drive_dir, exist_ok=True)
print("Drive monté :", os.path.ismount('/content/drive'))

Mounted at /content/drive
Drive monté : True


In [8]:
!ls -lh /content/drive/MyDrive/otto_project

total 0


In [10]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [11]:
import os

# 1. Configurer ton jeton Kaggle directement en texte
os.makedirs('/root/.kaggle', exist_ok=True)
token_content = "KGAT_faa83071852035404defa53f96aae3d9"

with open('/root/.kaggle/access_token', 'w') as f:
    f.write(token_content)

os.chmod('/root/.kaggle/access_token', 0o600)
print("✅ Jeton Kaggle configuré avec succès !")

# 2. Télécharger le dataset OTTO directement à haute vitesse
print("⏳ Téléchargement du dataset OTTO en cours...")
!kaggle competitions download -c otto-recommender-system
print("🎉 Téléchargement terminé !")

✅ Jeton Kaggle configuré avec succès !
⏳ Téléchargement du dataset OTTO en cours...
otto-recommender-system.zip: Skipping, found more recently modified local copy (use --force to force download)
🎉 Téléchargement terminé !


In [13]:
import zipfile
import os

# 1. Définir le dossier de destination pour l'extraction
extract_path = '/content/dataset_otto'
os.makedirs(extract_path, exist_ok=True)

# 2. Extraire le fichier ZIP
print("📦 Extraction du fichier ZIP en cours...")
with zipfile.ZipFile('otto-recommender-system.zip', 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print("🚀 Extraction terminée ! Voici les fichiers disponibles :")
print(os.listdir(extract_path))

📦 Extraction du fichier ZIP en cours...
🚀 Extraction terminée ! Voici les fichiers disponibles :
['train.jsonl', 'sample_submission.csv', 'test.jsonl']


In [14]:

import json

data_dir = '/content/dataset_otto'

with open(f'{data_dir}/train.jsonl', 'r') as f:
    for i in range(3):
        s = json.loads(f.readline())
        print(f"TRAIN Session {s['session']} | nb events: {len(s['events'])}")
        print("  3 premiers events:", s['events'][:3])

with open(f'{data_dir}/test.jsonl', 'r') as f:
    for i in range(3):
        s = json.loads(f.readline())
        print(f"TEST  Session {s['session']} | nb events: {len(s['events'])}")
        print("  3 derniers events:", s['events'][-3:])

TRAIN Session 0 | nb events: 276
  3 premiers events: [{'aid': 1517085, 'ts': 1659304800025, 'type': 'clicks'}, {'aid': 1563459, 'ts': 1659304904511, 'type': 'clicks'}, {'aid': 1309446, 'ts': 1659367439426, 'type': 'clicks'}]
TRAIN Session 1 | nb events: 32
  3 premiers events: [{'aid': 424964, 'ts': 1659304800025, 'type': 'carts'}, {'aid': 1492293, 'ts': 1659304852871, 'type': 'clicks'}, {'aid': 1492293, 'ts': 1659304863627, 'type': 'carts'}]
TRAIN Session 2 | nb events: 33
  3 premiers events: [{'aid': 763743, 'ts': 1659304800038, 'type': 'clicks'}, {'aid': 137492, 'ts': 1659304866606, 'type': 'clicks'}, {'aid': 504789, 'ts': 1659304915207, 'type': 'clicks'}]
TEST  Session 12899779 | nb events: 1
  3 derniers events: [{'aid': 59625, 'ts': 1661724000278, 'type': 'clicks'}]
TEST  Session 12899780 | nb events: 5
  3 derniers events: [{'aid': 973453, 'ts': 1661724109199, 'type': 'clicks'}, {'aid': 736515, 'ts': 1661724136868, 'type': 'clicks'}, {'aid': 1142000, 'ts': 1661724155248, 'type

In [15]:
import pandas as pd

sub = pd.read_csv(f'{data_dir}/sample_submission.csv', nrows=10)
print(sub.head())

      session_type                labels
0  12899779_clicks  129004 126836 118524
1   12899779_carts  129004 126836 118524
2  12899779_orders  129004 126836 118524
3  12899780_clicks  129004 126836 118524
4   12899780_carts  129004 126836 118524


In [17]:
import json
import os
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
from tqdm import tqdm

data_dir = '/content/dataset_otto'
parquet_dir = '/content/drive/MyDrive/otto_project/otto_parquet'
os.makedirs(parquet_dir, exist_ok=True)

type_map = {'clicks': 0, 'carts': 1, 'orders': 2}

def jsonl_to_parquet(jsonl_path, out_parquet_path, chunk_size=100_000):
    if os.path.exists(out_parquet_path):
        os.remove(out_parquet_path)
    writer = None
    rows = []

    with open(jsonl_path, 'r') as f:
        for line in tqdm(f, desc=f"Conversion {os.path.basename(jsonl_path)}"):
            obj = json.loads(line)
            sess = obj['session']
            for ev in obj['events']:
                rows.append((
                    sess,
                    ev['aid'],
                    ev['ts'],
                    type_map[ev['type']]
                ))

            if len(rows) >= chunk_size * 15:
                df = pd.DataFrame(rows, columns=['session', 'aid', 'ts', 'type'])
                df['session'] = df['session'].astype('int32')
                df['aid'] = df['aid'].astype('int32')
                df['ts'] = df['ts'].astype('int64')
                df['type'] = df['type'].astype('uint8')

                table = pa.Table.from_pandas(df, preserve_index=False)
                if writer is None:
                    writer = pq.ParquetWriter(out_parquet_path, table.schema, compression='snappy')
                writer.write_table(table)
                rows.clear()

    if rows:
        df = pd.DataFrame(rows, columns=['session', 'aid', 'ts', 'type'])
        df['session'] = df['session'].astype('int32')
        df['aid'] = df['aid'].astype('int32')
        df['ts'] = df['ts'].astype('int64')
        df['type'] = df['type'].astype('uint8')
        table = pa.Table.from_pandas(df, preserve_index=False)
        if writer is None:
            writer = pq.ParquetWriter(out_parquet_path, table.schema, compression='snappy')
        writer.write_table(table)

    if writer is not None:
        writer.close()

jsonl_to_parquet(f'{data_dir}/train.jsonl', f'{parquet_dir}/train.parquet')
jsonl_to_parquet(f'{data_dir}/test.jsonl', f'{parquet_dir}/test.parquet')
print("✅ Conversion Parquet terminée !")

Conversion train.jsonl: 12899779it [13:35, 15811.27it/s]
Conversion test.jsonl: 1671803it [00:37, 44657.72it/s]


✅ Conversion Parquet terminée !


In [20]:
!ls -lh /content/drive/MyDrive/otto_project/otto_parquet

total 2.1G
-rw------- 1 root root  74M Nov 24 19:26 test.parquet
-rw------- 1 root root 2.1G Nov 24 19:25 train.parquet


In [22]:
import shutil
import os

os.makedirs('/content/drive/MyDrive/otto_project', exist_ok=True)
!cp -r /content/otto_parquet /content/drive/MyDrive/otto_project/
print("✅ Parquet sauvegardé sur Google Drive !")

cp: cannot stat '/content/otto_parquet': No such file or directory
✅ Parquet sauvegardé sur Google Drive !


In [23]:
import numpy as np
import pandas as pd
import pyarrow.parquet as pq

parquet_dir = '/content/drive/MyDrive/otto_project/otto_parquet'
train_path = f'{parquet_dir}/train.parquet'

# 1. Trouver le timestamp max sans charger tout le fichier
ts_col = pq.read_table(train_path, columns=['ts'])['ts'].to_numpy()
max_ts = int(ts_col.max())
del ts_col

scale = 1000 if max_ts > 1e11 else 1
val_cutoff = max_ts - 2 * 86400 * scale  # 2 derniers jours

# 2. Charger uniquement les lignes récentes pour la validation
val_df = pq.read_table(
    train_path,
    columns=['session', 'aid', 'ts', 'type'],
    filters=[('ts', '>=', val_cutoff)]
).to_pandas()

# Échantillonner 20% des sessions de validation pour aller vite et économiser la RAM
rng = np.random.default_rng(42)
unique_val_sess = val_df['session'].unique()
sampled_sess = set(rng.choice(unique_val_sess, size=max(1, int(len(unique_val_sess) * 0.2)), replace=False))
val_df = val_df[val_df['session'].isin(sampled_sess)].copy()

# Garder les sessions avec >= 4 événements et couper à 70%
sess_counts = val_df.groupby('session')['aid'].count()
valid_sessions = set(sess_counts[sess_counts >= 4].index)
val_df = val_df[val_df['session'].isin(valid_sessions)].sort_values(['session', 'ts'])

val_df['rank'] = val_df.groupby('session').cumcount()
val_df['cnt'] = val_df.groupby('session')['aid'].transform('count')
cut_idx = np.maximum(1, (val_df['cnt'] * 0.7).astype(int))

val_hist = val_df[val_df['rank'] < cut_idx][['session', 'aid', 'ts', 'type']]
val_fut = val_df[val_df['rank'] >= cut_idx]

gt_clicks = val_fut[val_fut['type'] == 0].groupby('session')['aid'].first().to_dict()
gt_carts  = val_fut[val_fut['type'] == 1].groupby('session')['aid'].apply(set).to_dict()
gt_orders = val_fut[val_fut['type'] == 2].groupby('session')['aid'].apply(set).to_dict()

def score_recall20(preds_clicks, preds_carts, preds_orders):
    c_hits, c_total = 0, 0
    for s, target in gt_clicks.items():
        c_total += 1
        if target in preds_clicks.get(s, [])[:20]:
            c_hits += 1
    r_clicks = c_hits / max(c_total, 1)

    ca_hits, ca_denom = 0, 0
    for s, targets in gt_carts.items():
        if not targets:
            continue
        ca_hits += len(targets & set(preds_carts.get(s, [])[:20]))
        ca_denom += min(len(targets), 20)
    r_carts = ca_hits / max(ca_denom, 1)

    o_hits, o_denom = 0, 0
    for s, targets in gt_orders.items():
        if not targets:
            continue
        o_hits += len(targets & set(preds_orders.get(s, [])[:20]))
        o_denom += min(len(targets), 20)
    r_orders = o_hits / max(o_denom, 1)

    total = 0.10 * r_clicks + 0.30 * r_carts + 0.60 * r_orders
    return total, r_clicks, r_carts, r_orders

print(f"✅ Validation prête : {len(gt_clicks)} sessions cibles.")

✅ Validation prête : 203452 sessions cibles.


In [27]:
import os, pickle

artifacts_dir = '/content/drive/MyDrive/otto_project/artifacts'
os.makedirs(artifacts_dir, exist_ok=True)

with open(f'{artifacts_dir}/covis_val.pkl', 'wb') as f:
    pickle.dump({'covis': covis_val, 'fallback': fallback_val}, f)

print("✅ covis_val.pkl sauvegardé sur Google Drive !")

✅ covis_val.pkl sauvegardé sur Google Drive !


In [24]:
from collections import defaultdict, Counter
from tqdm import tqdm

def build_covis_streamed(parquet_file, excluded_sessions, window_size=12, top_k=20, batch_size=500_000):
    type_weights = {0: 1.0, 1: 3.0, 2: 6.0}
    covis = defaultdict(Counter)
    pop_counter = Counter()

    pf = pq.ParquetFile(parquet_file)
    for batch in tqdm(pf.iter_batches(batch_size=batch_size, columns=['session', 'aid', 'ts', 'type']), desc="Co-visitation"):
        df_b = batch.to_pandas()
        if excluded_sessions:
            df_b = df_b[~df_b['session'].isin(excluded_sessions)]
        if df_b.empty:
            continue

        pop_counter.update(df_b['aid'].tolist())
        df_b = df_b.sort_values(['session', 'ts'])

        for _, grp in df_b.groupby('session', sort=False):
            aids = grp['aid'].tolist()
            types = grp['type'].tolist()
            n = len(aids)
            for i in range(n):
                a1 = aids[i]
                end = min(n, i + 1 + window_size)
                for j in range(i + 1, end):
                    a2 = aids[j]
                    if a1 != a2:
                        covis[a1][a2] += type_weights[types[j]] / (j - i)

        # Pruning régulier pour garder la mémoire sous contrôle
        if len(covis) > 250_000:
            for k in list(covis.keys()):
                if len(covis[k]) > top_k * 2:
                    covis[k] = Counter(dict(covis[k].most_common(top_k)))

    top_covis = {k: [aid for aid, _ in cnt.most_common(top_k)] for k, cnt in covis.items()}
    fallback = [aid for aid, _ in pop_counter.most_common(30)]
    return top_covis, fallback

covis_val, fallback_val = build_covis_streamed(train_path, excluded_sessions=valid_sessions)
print(f"✅ Matrice construite pour {len(covis_val)} articles.")

Co-visitation: 434it [1:34:05, 13.01s/it]


✅ Matrice construite pour 1852893 articles.


In [34]:
def recommend_for_sessions(hist_df, covis_map, fallback_aids, top_n=20):
    preds = {}
    grouped = hist_df.sort_values(['session', 'ts']).groupby('session', sort=False)['aid'].apply(list)
    for sess, aids in grouped.items():
        seen = set()
        rec = []

        for a in reversed(aids):
            if a not in seen:
                seen.add(a)
                rec.append(a)
                if len(rec) == top_n:
                    break

        if len(rec) < top_n:
            for a in reversed(aids[-5:]):
                for cand in covis_map.get(a, []):
                    if cand not in seen:
                        seen.add(cand)
                        rec.append(cand)
                        if len(rec) == top_n:
                            break
                if len(rec) == top_n:
                    break

        if len(rec) < top_n:
            for fb in fallback_aids:
                if fb not in seen:
                    seen.add(fb)
                    rec.append(fb)
                    if len(rec) == top_n:
                        break

        preds[sess] = rec[:top_n]
    return preds

p_c = recommend_for_sessions(val_hist, covis_val, fallback_val)
p_ca = recommend_for_sessions(val_hist, covis_val, fallback_val)
p_o = recommend_for_sessions(val_hist, covis_val, fallback_val)

tot, rc, rca, ro = score_recall20(p_c, p_ca, p_o)
print(f"📊 Score validation Recall@20 : {tot:.5f} (clicks={rc:.5f}, carts={rca:.5f}, orders={ro:.5f})")

# Génération de la soumission sur test.parquet
test_df = pq.read_table(f'{parquet_dir}/test.parquet').to_pandas()
covis_full, fallback_full = build_covis_streamed(train_path, excluded_sessions=None)

t_c = recommend_for_sessions(test_df, covis_full, fallback_full)
t_ca = recommend_for_sessions(test_df, covis_full, fallback_full)
t_o = recommend_for_sessions(test_df, covis_full, fallback_full)

sub_rows = []
for sess in test_df['session'].unique():
    sub_rows.append((f"{sess}_clicks", " ".join(map(str, t_c[sess][:20]))))
    sub_rows.append((f"{sess}_carts",  " ".join(map(str, t_ca[sess][:20]))))
    sub_rows.append((f"{sess}_orders", " ".join(map(str, t_o[sess][:20]))))

sub_df = pd.DataFrame(sub_rows, columns=['session_type', 'labels'])
sub_out = '/content/drive/MyDrive/otto_project/submission.csv'
sub_df.to_csv(sub_out, index=False)
print(f"✅ Soumission enregistrée : {sub_out} ({len(sub_df)} lignes)")

📊 Score validation Recall@20 : 0.63479 (clicks=0.46207, carts=0.47306, orders=0.74444)


Co-visitation: 434it [1:34:27, 13.06s/it]


✅ Soumission enregistrée : /content/drive/MyDrive/otto_project/submission.csv (5015409 lignes)


In [1]:
import glob
import os

print("Mountpoint actif ?", os.path.ismount('/content/drive'))
print("Fichiers submission.csv trouvés :", glob.glob('/content/**/submission.csv', recursive=True))
print("Autres .csv dans /content :", glob.glob('/content/**/*.csv', recursive=True))

Mountpoint actif ? False
Fichiers submission.csv trouvés : []
Autres .csv dans /content : ['/content/sample_data/california_housing_train.csv', '/content/sample_data/mnist_test.csv', '/content/sample_data/mnist_train_small.csv', '/content/sample_data/california_housing_test.csv']


In [12]:
!ls -lh /content/otto-recommender-system.zip

-rw-r--r-- 1 root root 2.0G Nov  1  2022 /content/otto-recommender-system.zip


In [21]:
!ls -lh /content/drive/MyDrive/otto_project/otto_parquet

total 2.1G
-rw------- 1 root root  74M Nov 24 19:26 test.parquet
-rw------- 1 root root 2.1G Nov 24 19:25 train.parquet


In [31]:
print("covis_val existe ?", 'covis_val' in globals())
print("covis_full existe ?", 'covis_full' in globals())

covis_val existe ? True
covis_full existe ? False


In [28]:
!ls -lh /content/drive/MyDrive/otto_project/artifacts/covis_val.pkl

-rw------- 1 root root 186M Nov 24 21:11 /content/drive/MyDrive/otto_project/artifacts/covis_val.pkl


In [29]:
ls -lh /content/drive/MyDrive/otto_project /content/drive/MyDrive/otto_project/otto_parquet /content/drive/MyDrive/otto_project/artifacts

/content/drive/MyDrive/otto_project:
total 8.0K
drwx------ 2 root root 4.0K Nov 24 21:11 artifacts/
drwx------ 2 root root 4.0K Nov 24 19:26 otto_parquet/

/content/drive/MyDrive/otto_project/artifacts:
total 186M
-rw------- 1 root root    0 Nov 24 21:10 covis_full.pkl
-rw------- 1 root root 186M Nov 24 21:11 covis_val.pkl

/content/drive/MyDrive/otto_project/otto_parquet:
total 2.1G
-rw------- 1 root root  74M Nov 24 19:26 test.parquet
-rw------- 1 root root 2.1G Nov 24 19:25 train.parquet


In [33]:
!rm -f /content/drive/MyDrive/otto_project/artifacts/covis_full.pkl
!ls -lh /content/drive/MyDrive/otto_project/artifacts

total 186M
-rw------- 1 root root 186M Nov 24 21:11 covis_val.pkl


In [35]:
import os
import pickle

artifacts_dir = '/content/drive/MyDrive/otto_project/artifacts'
os.makedirs(artifacts_dir, exist_ok=True)

if 'covis_full' in globals() and 'fallback_full' in globals():
    with open(f'{artifacts_dir}/covis_full.pkl', 'wb') as f:
        pickle.dump({'covis': covis_full, 'fallback': fallback_full}, f)
    print("✅ covis_full.pkl sauvegardé sur Google Drive !")
else:
    print("ℹ️ covis_full n'est plus en mémoire (submission.csv est déjà généré).")

✅ covis_full.pkl sauvegardé sur Google Drive !


In [36]:
!ls -lh /content/drive/MyDrive/otto_project/artifacts

total 373M
-rw------- 1 root root 187M Nov 24 23:05 covis_full.pkl
-rw------- 1 root root 186M Nov 24 21:11 covis_val.pkl


In [38]:
!ls -lh /content/drive/MyDrive/otto_project/submission.csv
!wc -l /content/drive/MyDrive/otto_project/submission.csv
!head -n 5 /content/drive/MyDrive/otto_project/submission.csv
!tail -n 5 /content/drive/MyDrive/otto_project/submission.csv

-rw------- 1 root root 783M Nov 24 23:03 /content/drive/MyDrive/otto_project/submission.csv
5015410 /content/drive/MyDrive/otto_project/submission.csv
session_type,labels
12899779_clicks,59625 737445 469285 1340695 1246235 894169 1804863 499621 941596 601769 731692 448688 1700255 1660529 608805 729915 1223875 1401514 514164 1354960
12899779_carts,59625 737445 469285 1340695 1246235 894169 1804863 499621 941596 601769 731692 448688 1700255 1660529 608805 729915 1223875 1401514 514164 1354960
12899779_orders,59625 737445 469285 1340695 1246235 894169 1804863 499621 941596 601769 731692 448688 1700255 1660529 608805 729915 1223875 1401514 514164 1354960
12899780_clicks,1142000 736515 973453 582732 1344758 1502122 487136 889686 1515511 768323 1032776 636101 1419849 1263108 1383529 258814 595830 819422 1717153 103974
14571580_carts,202353 682237 433425 477239 336919 891417 1314576 925638 985380 886121 773782 1295081 387358 590739 1682397 862690 32322 1627186 78785 634205
14571580_orders,202

In [39]:
!ls -lh /content/drive/MyDrive/otto_project/artifacts

total 373M
-rw------- 1 root root 187M Nov 24 23:05 covis_full.pkl
-rw------- 1 root root 186M Nov 24 21:11 covis_val.pkl
